In [0]:
from pyspark.sql.functions import current_timestamp, lit
import pandas as pd

In [0]:
BASE_PATH = "/Volumes/workspace/project_data_football_raw/mercado_raw"

In [0]:
# Lista os arquivos no diretório BASE_PATH
arquivos = dbutils.fs.ls(BASE_PATH)

# Ordena os arquivos pelo nome em ordem decrescente
arquivos = sorted(arquivos, key=lambda x: x.name, reverse=True)

# Verifica se há arquivos disponíveis
if not arquivos:
    raise Exception("Nenhum arquivo encontrado na landing de mercado")

# Seleciona o caminho do arquivo mais recente
ultimo_arquivo = arquivos[0].path

# Exibe o caminho do arquivo selecionado
print(f"Arquivo: {ultimo_arquivo}")

In [0]:
# LEITURA DA LANDING: lê o arquivo JSON mais recente da landing zone
df_raw = spark.read.json(ultimo_arquivo)

# Converte o DataFrame Spark para um dicionário Python
mercado = df_raw.toPandas().iloc[0].to_dict()

# Obtém a rodada atual do mercado
rodada_atual = mercado.get("rodada_atual")

# Obtém o status do mercado
status_mercado = mercado.get("status_mercado")

# Exibe a rodada de referência
print(f"Rodada referência: {rodada_atual}")

# Exibe o status do mercado
print(f"Status mercado: {status_mercado}")

In [0]:
# CLUBES

# Obtém o dicionário de clubes do mercado
clubes = mercado.get("clubes", {})

# Exibe a quantidade de clubes encontrados
print("Quantidade de clubes:", len(clubes))

if clubes:
    lista_clubes = []

    # Itera sobre cada clube, adicionando o clube_id ao dicionário de dados
    for clube_id, dados in clubes.items():
        dados["clube_id"] = int(clube_id)
        lista_clubes.append(dados)

    # Cria um DataFrame Spark a partir da lista de clubes
    df_clubes = spark.createDataFrame(pd.DataFrame(lista_clubes)) \
        .withColumn("rodada_referencia", lit(rodada_atual).cast("int")) \
        .withColumn("dt_ingestao", current_timestamp())

    # Grava o DataFrame na tabela Delta bronze de clubes
    df_clubes.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("project_data_football_bronze.clubes")

    # Confirma o carregamento dos clubes
    print("Clubes carregados com sucesso!")

In [0]:
# POSICOES

# Obtém o dicionário de posições do mercado
posicoes = mercado.get("posicoes", {})

# Exibe a quantidade de posições encontradas
print("Quantidade de posicoes:", len(posicoes))

if posicoes:
    lista_posicoes = []

    # Itera sobre cada posição, adicionando o posicao_id ao dicionário de dados
    for pos_id, dados in posicoes.items():
        dados["posicao_id"] = int(pos_id)
        lista_posicoes.append(dados)

    # Cria um DataFrame Spark a partir da lista de posições
    df_posicoes = spark.createDataFrame(pd.DataFrame(lista_posicoes)) \
        .withColumn("rodada_referencia", lit(rodada_atual).cast("int")) \
        .withColumn("dt_ingestao", current_timestamp())

    # Grava o DataFrame na tabela Delta bronze de posições
    df_posicoes.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("project_data_football_bronze.posicoes")

    # Confirma o carregamento das posições
    print("Posições carregadas com sucesso!")

In [0]:
# ATLETAS

# Obtém a lista de atletas do mercado
atletas = mercado.get("atletas", [])

# Exibe a quantidade de atletas encontrados
print("Quantidade de atletas:", len(atletas))

# Verifica se a lista de atletas está preenchida
if isinstance(atletas, list) and len(atletas) > 0:

    # Cria uma lista de dicionários para cada atleta
    lista_atletas = [dict(a) for a in atletas]

    # Cria um DataFrame Spark a partir da lista de atletas e adiciona colunas de referência e ingestão
    df_atletas = spark.createDataFrame(pd.DataFrame(lista_atletas)) \
        .withColumn("rodada_referencia", lit(rodada_atual).cast("int")) \
        .withColumn("dt_ingestao", current_timestamp())

    # Grava o DataFrame na tabela Delta bronze de atletas
    df_atletas.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("project_data_football_bronze.atletas")

In [0]:
print("Atletas carregados com sucesso!")